# Introduction

This is a solution fo IEEE-CIS Fraud detection competition.
In order to be easily to adapt to various environments, we structured this Notebook so that we parameterize the data loading and we can easily refactor the code to run as a Python module, not necesarly as a Jupyter Notebook.

The main objective of this Notebook is to illustrate feature engineering techniques for a fraud detection problem solution.

# Import packages

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression


# Data loading utilities

In [2]:
@dataclass(frozen=True)
class IEEECISPaths:
    train_transaction: Path
    train_identity: Path
    test_transaction: Path
    test_identity: Path


def load_ieee_cis(paths: IEEECISPaths) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    """
    Loads IEEE-CIS tables and joins identity to transaction on TransactionID.

    Returns:
        X_full: training features (joined)
        y:      training labels (isFraud)
        X_test: test features (joined)
    """
    train_tr = pd.read_csv(paths.train_transaction)
    train_id = pd.read_csv(paths.train_identity)
    test_tr = pd.read_csv(paths.test_transaction)
    test_id = pd.read_csv(paths.test_identity)

    # Left-join identity onto transaction (not all TransactionID values appear in identity)
    train = train_tr.merge(train_id, on="TransactionID", how="left")
    test = test_tr.merge(test_id, on="TransactionID", how="left")

    if "isFraud" not in train.columns:
        raise ValueError("Expected 'isFraud' in training data.")

    y = train["isFraud"].astype(int)
    X_full = train.drop(columns=["isFraud"])

    return X_full, y, test


# Feature engineering

In [3]:
def _safe_str_series(s: pd.Series) -> pd.Series:
    """Ensure string dtype without converting NaN to 'nan' strings."""
    out = s.astype("string")
    return out


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering applied identically to train/valid/test.

    This function must NOT use target labels.
    It must be deterministic and operate only on df columns.

    The IEEE-CIS dataset includes many anonymized columns; we focus on robust, typical signals:
    - Time decomposition from TransactionDT
    - Amount transforms
    - Email domain parsing
    - Card-based composite keys (frequency-style proxies without using target)
    - Address / device normalization
    """
    df = df.copy()

    # ---- Time features ----
    # In IEEE-CIS, TransactionDT is a time delta (seconds) from a reference moment.
    # We extract coarse periodicities that often help fraud models.
    if "TransactionDT" in df.columns:
        dt = df["TransactionDT"].astype("float64")
        df["TransactionDT_days"] = dt / (3600.0 * 24.0)
        df["TransactionDT_hours"] = dt / 3600.0

        # Cyclical components for time-of-day proxy (coarse; assumes 24h periodicity)
        # We cannot infer actual timezone; this is still useful as a periodic signal.
        hour_of_day = (dt / 3600.0) % 24.0
        df["hour_sin"] = np.sin(2.0 * np.pi * hour_of_day / 24.0)
        df["hour_cos"] = np.cos(2.0 * np.pi * hour_of_day / 24.0)

        # Day-of-week proxy (7-day periodicity)
        day_of_week = (dt / (3600.0 * 24.0)) % 7.0
        df["dow_sin"] = np.sin(2.0 * np.pi * day_of_week / 7.0)
        df["dow_cos"] = np.cos(2.0 * np.pi * day_of_week / 7.0)

    # ---- Amount transforms ----
    if "TransactionAmt" in df.columns:
        amt = df["TransactionAmt"].astype("float64")
        df["TransactionAmt_log1p"] = np.log1p(amt.clip(lower=0))
        # Fractional cents sometimes carry signal; keep it as a separate feature
        df["TransactionAmt_frac"] = (amt - np.floor(amt)).fillna(0.0)

    # ---- Email domain parsing ----
    # Many solutions use P_emaildomain / R_emaildomain.
    # We derive top-level domain and a "same domain" indicator.
    def extract_tld(email_domain: pd.Series) -> pd.Series:
        s = _safe_str_series(email_domain)
        # e.g., "gmail.com" -> "com", "yahoo.co.uk" -> "uk"
        tld = s.str.split(".").str[-1]
        return tld

    if "P_emaildomain" in df.columns:
        df["P_emaildomain_tld"] = extract_tld(df["P_emaildomain"])
    if "R_emaildomain" in df.columns:
        df["R_emaildomain_tld"] = extract_tld(df["R_emaildomain"])
    if "P_emaildomain" in df.columns and "R_emaildomain" in df.columns:
        p = _safe_str_series(df["P_emaildomain"])
        r = _safe_str_series(df["R_emaildomain"])
        df["email_domain_match"] = (p == r).astype("Int64")

    # ---- Composite identifiers (unsupervised) ----
    # Fraud patterns often depend on combinations of card/address/device.
    # We build stable composite keys as categorical features.
    card_cols = [c for c in ["card1", "card2", "card3", "card4", "card5", "card6"] if c in df.columns]
    addr_cols = [c for c in ["addr1", "addr2"] if c in df.columns]

    if card_cols:
        # Create a compact composite card signature.
        df["card_sig"] = (
            df[card_cols]
            .astype("string")
            .fillna("NA")
            .agg("_".join, axis=1)
        )

    if addr_cols:
        df["addr_sig"] = (
            df[addr_cols]
            .astype("string")
            .fillna("NA")
            .agg("_".join, axis=1)
        )

    # Combine card + address as a higher-order key when available.
    if "card_sig" in df.columns and "addr_sig" in df.columns:
        df["card_addr_sig"] = (df["card_sig"].astype("string") + "|" + df["addr_sig"].astype("string"))

    # ---- Device / identity normalization ----
    # DeviceType and DeviceInfo are often messy strings.
    for col in ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_33"]:
        if col in df.columns:
            s = _safe_str_series(df[col]).str.lower()
            # Keep only first token (often brand / OS) as a coarse normalization
            df[col + "_coarse"] = s.str.split().str[0]

    # ---- Count-encoding (frequency proxy) WITHOUT target leakage ----
    # Count encoding can be fit on training only; however, this function is called for train/valid/test.
    # To keep this function stateless and safe, we do not compute global counts here.
    # Instead, the correct way is to implement count-encoding as a transformer fitted on train only.
    #
    # We still leave composite signatures in place (card_sig, card_addr_sig, etc.)
    # so a dedicated transformer can count-encode them later if desired.

    # Clean up: avoid accidental object dtype explosions
    # (We intentionally keep categorical engineered columns as string dtype.)
    return df


## Optional - count encoder

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class CountEncoder(BaseEstimator, TransformerMixin):
    """
    Count-encodes specified categorical columns by replacing each category with its frequency
    computed on the training fold only.

    This is a supervised-safe (target-free) encoding but it must still be fit on train only
    to avoid peeking at validation distribution.
    """
    def __init__(self, cols: List[str]):
        self.cols = cols
        self.maps_ = {}

    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()
        self.maps_ = {}
        for c in self.cols:
            if c in X.columns:
                vc = X[c].astype("string").value_counts(dropna=False)
                self.maps_[c] = vc
        return self

    def transform(self, X: pd.DataFrame):
        X = X.copy()
        for c, vc in self.maps_.items():
            if c in X.columns:
                # Map to counts; unseen categories -> 0
                X[c + "_count"] = X[c].astype("string").map(vc).fillna(0).astype("int64")
        return X



# Build pipeline

In [5]:
def build_pipeline(X_example: pd.DataFrame) -> Pipeline:
    """
    Builds an end-to-end pipeline:
    - Feature engineering (custom function)
    - Count encoding for a small set of engineered high-signal keys
    - ColumnTransformer: impute + encode categoricals, impute numericals
    - Classifier: logistic regression with class_weight to handle imbalance

    Notes:
    - Logistic regression is chosen for clarity and as a strong baseline.
    - For higher performance, you would typically use gradient boosting;
      the preprocessing strategy remains the same.
    """
    # Identify likely categorical columns (object/string/category) after feature engineering
    # We must do feature engineering first to see engineered columns.
    X_fe = engineer_features(X_example)

    # Choose a small set of columns for count encoding (common in IEEE-CIS solutions)
    count_cols = [c for c in ["card_sig", "addr_sig", "card_addr_sig"] if c in X_fe.columns]

    # Recompute dtypes after count encoding will add numeric columns; that is fine.
    # Define which columns are treated as categorical vs numerical for ColumnTransformer.
    cat_cols = [c for c in X_fe.columns if str(X_fe[c].dtype) in ("object", "string", "category")]
    num_cols = [c for c in X_fe.columns if c not in cat_cols]

    # Preprocessing for numeric features
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        # Standardization is optional for logistic regression; often helps.
        # If you replace the model with a tree ensemble, you can drop scaling.
        # ("scaler", StandardScaler()),
    ])

    # Preprocessing for categorical features
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ],
        remainder="drop"
    )

    model = LogisticRegression(
        max_iter=3000,
        solver="saga",
        n_jobs=None,
        class_weight="balanced"  # handles extreme imbalance without resampling
    )

    # We implement feature engineering and count encoding as pipeline steps.
    # This ensures they are applied consistently and in the correct order.
    pipe = Pipeline(steps=[
        ("feature_engineering", FunctionTransformer(engineer_features, validate=False)),
        ("count_encoding", CountEncoder(cols=count_cols)),
        ("preprocess", preprocessor),
        ("model", model),
    ])

    return pipe

# Training and evaluation

In [6]:
from sklearn.preprocessing import FunctionTransformer

def train_validate(
    X: pd.DataFrame,
    y: pd.Series,
    random_state: int = 42
) -> Pipeline:
    """
    Train/validation split, model fit, and evaluation with ROC-AUC and PR-AUC.

    PR-AUC (Average Precision) is informative under extreme imbalance.
    """
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y,
        test_size=0.2,
        random_state=random_state,
        stratify=y
    )

    pipe = build_pipeline(X_train)

    pipe.fit(X_train, y_train)

    # Predicted probabilities for ranking metrics
    valid_proba = pipe.predict_proba(X_valid)[:, 1]

    roc = roc_auc_score(y_valid, valid_proba)
    pr = average_precision_score(y_valid, valid_proba)

    print(f"Validation ROC-AUC: {roc:.6f}")
    print(f"Validation PR-AUC (Average Precision): {pr:.6f}")

    return pipe



## Predict test 

In [7]:
def predict_test(pipe: Pipeline, X_test: pd.DataFrame) -> np.ndarray:
    """
    Produces predicted probabilities for the Kaggle test set.
    """
    return pipe.predict_proba(X_test)[:, 1]


# Main loop

In [8]:
def main():
    """
    Example usage.

    Data acquisition (one-time):
    - Option A: Download manually from Kaggle competition page and unzip locally.
    - Option B: Kaggle API:
        pip install kaggle
        kaggle competitions download -c ieee-fraud-detection
        unzip ieee-fraud-detection.zip -d ./ieee_cis/

    Then set data_dir accordingly.
    """
    data_dir = Path("/kaggle/input/competitions/ieee-fraud-detection")  # <-- change to your local path

    paths = IEEECISPaths(
        train_transaction=data_dir / "train_transaction.csv",
        train_identity=data_dir / "train_identity.csv",
        test_transaction=data_dir / "test_transaction.csv",
        test_identity=data_dir / "test_identity.csv",
    )

    X, y, X_test = load_ieee_cis(paths)

    # Fit and evaluate
    pipe = train_validate(X, y, random_state=42)

    # Predict test for submission
    test_proba = predict_test(pipe, X_test)

    # Build Kaggle submission file: TransactionID + isFraud
    # Kaggle expects TransactionID from test_transaction.csv
    test_tr = pd.read_csv(paths.test_transaction, usecols=["TransactionID"])
    submission = pd.DataFrame({
        "TransactionID": test_tr["TransactionID"],
        "isFraud": test_proba
    })

    out_path = data_dir / "submission.csv"
    submission.to_csv(out_path, index=False)
    print(f"Saved submission to: {out_path}")

# Run

In [9]:
if __name__ == "__main__":
    main()

TypeError: boolean value of NA is ambiguous